In [3]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

df_scaled = pd.read_csv('dataset_cleaned.csv')
cols_to_drop = [col for col in df_scaled.columns if 'Cluster' in col]
X = df_scaled.drop(columns=cols_to_drop)
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X)

dbscan = DBSCAN(eps=1.2, min_samples=3)
dbscan_labels = dbscan.fit_predict(X_pca)

def evaluate_clustering(data, labels, model_name):
    unique_labels = set(labels)
    noise_points = list(labels).count(-1)
    valid_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    
   
    if len(unique_labels) > 1:
        sil = silhouette_score(data, labels)
        db = davies_bouldin_score(data, labels)
        ch = calinski_harabasz_score(data, labels)
    else:
        sil, db, ch = None, None, None

    return {
        "Model": model_name,
        "Valid Clusters": valid_clusters,
        "Noise Points": noise_points,
        "Silhouette Score ": round(sil, 4) if sil is not None else "N/A",
        "Davies-Bouldin ": round(db, 4) if db is not None else "N/A",
        "Calinski-Harabasz ": round(ch, 4) if ch is not None else "N/A"
    }


kmeans_metrics = evaluate_clustering(X, kmeans_labels, "K-Means (k=5)")
dbscan_metrics = evaluate_clustering(X_pca, dbscan_labels, "PCA-DBSCAN (eps=1.2, min_pts=3)")

benchmark_df = pd.DataFrame([kmeans_metrics, dbscan_metrics])

print("\n" + "="*80)
print("CLUSTERING ALGORITHM BENCHMARK REPORT  ")
print("="*80)
print(benchmark_df.to_string(index=False)) 
print("="*80)


CLUSTERING ALGORITHM BENCHMARK REPORT  
                          Model  Valid Clusters  Noise Points  Silhouette Score   Davies-Bouldin   Calinski-Harabasz 
                  K-Means (k=5)               5             0             0.1243           1.9146            274.9170
PCA-DBSCAN (eps=1.2, min_pts=3)               2            14             0.2578           2.4026             10.1033


Since the number of clusters is lesser in DBSCAN hence the Silhouette Score is lesser but on rest of the indices which takes other factors into account Dbscan fails miserably

Thus K-Means was the optimal algorithm here